# Download Network Inputs from S3

Downloads step-03 network text assets from OpenAlex S3 into a local snapshot/query folder:
- edges.txt
- nodes_index.txt
- nodes.txt

Set `SNAPSHOT` and `QUERY` in the configuration cell before running.

## 1. Install and Import Required Libraries

In [ ]:
import boto3
from pathlib import Path

## 2. Set Configuration Variables

Edit `SNAPSHOT` and `QUERY` to match the dataset you want to download.

In [ ]:
# -- Change these two values before running ------------------------------------
SNAPSHOT = "2026-06-30"
QUERY    = "sustainability"
# ------------------------------------------------------------------------------

S3_BUCKET  = "openalex-results"
BASE_INPUT = Path("input") / f"snapshot_{SNAPSHOT}" / QUERY

S3_PREFIXES = {
    "edges":       f"snapshot_{SNAPSHOT}/queries/{QUERY}/network/edges.txt/",
    "nodes_index": f"snapshot_{SNAPSHOT}/queries/{QUERY}/network/nodes_index.txt/",
    "nodes":       f"snapshot_{SNAPSHOT}/queries/{QUERY}/network/nodes.txt/",
}

LOCAL_FILES = {
    "edges": BASE_INPUT / "edges.txt",
    "nodes_index": BASE_INPUT / "nodes_index.txt",
    "nodes": BASE_INPUT / "nodes.txt",
}

print(f"Snapshot: {SNAPSHOT}")
print(f"Query   : {QUERY}")
for name, prefix in S3_PREFIXES.items():
    print(f"S3 {name:11}: s3://{S3_BUCKET}/{prefix}")
for name, path in LOCAL_FILES.items():
    print(f"Local {name:8}: {path}")

S3 source : s3://openalex-outputs/cwts/sustainability/network_assets/version3/edges_bidirectional.txt/
Local dest: input/sustainability/version3/edges_bidirectional.txt


## 3. Create Local Snapshot/Query Directory

In [ ]:
BASE_INPUT.mkdir(parents=True, exist_ok=True)
print(f"Directory ready: {BASE_INPUT}")

Directory ready: input/sustainability/version3


## 4. Download Files from S3

Uses the default AWS credential chain (environment variables, `~/.aws/credentials`, or instance profile).

In [ ]:
s3 = boto3.client("s3")


def download_multipart_prefix(bucket: str, prefix: str, target: Path) -> None:
    paginator = s3.get_paginator("list_objects_v2")
    parts = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key == prefix or key.endswith("/"):
                continue
            parts.append(key)

    if not parts:
        raise FileNotFoundError(
            f"No objects found under s3://{bucket}/{prefix}\\n"
            "Check SNAPSHOT, QUERY and your AWS credentials."
        )

    parts.sort()
    with open(target, "wb") as out_f:
        for i, part_key in enumerate(parts, 1):
            print(f"  [{i}/{len(parts)}] {part_key.split('/')[-1]}", end="\\r")
            response = s3.get_object(Bucket=bucket, Key=part_key)
            for chunk in response["Body"].iter_chunks(chunk_size=8 * 1024 * 1024):
                out_f.write(chunk)
    print(f"  [done] {target.name} ({target.stat().st_size / (1024 ** 2):.2f} MB)")


for name in ["edges", "nodes_index", "nodes"]:
    prefix = S3_PREFIXES[name]
    target = LOCAL_FILES[name]
    print(f"Downloading {name} from s3://{S3_BUCKET}/{prefix}")
    download_multipart_prefix(S3_BUCKET, prefix, target)

print("All files downloaded.")

Found 72 part-file(s). Downloading and concatenating ...
  [72/72] run-1783156307537-part-r-00071
Download complete. Written to input/sustainability/version3/edges_bidirectional.txt


## 5. Verify Downloaded Files

In [ ]:
for name, local_file in LOCAL_FILES.items():
    if not local_file.exists():
        raise FileNotFoundError(f"Missing file: {local_file}")
    size_mb = local_file.stat().st_size / (1024 ** 2)
    print(f"{name:11} -> {local_file} ({size_mb:.2f} MB)")

✓ File found: input/sustainability/version3/edges_bidirectional.txt
  Size: 48.35 MB


## 6. Optional Integrity Check (No Dedup)

Step 03 already provides canonical network text assets, so dedup is not part of the default step-04 flow.
Run this only if you want quick diagnostics on the downloaded edge list.

In [ ]:
import pandas as pd

edges_file = LOCAL_FILES["edges"]

sample = pd.read_csv(
    edges_file,
    sep="\t",
    header=None,
    names=["src", "dst", "weight"],
    nrows=2_000_000,
    dtype={"src": int, "dst": int, "weight": float},
)

self_loops = int((sample["src"] == sample["dst"]).sum())
dupes = int(sample.duplicated(subset=["src", "dst", "weight"]).sum())

print(f"Sample rows      : {len(sample):,}")
print(f"Self-loops       : {self_loops:,}")
print(f"Exact duplicates : {dupes:,}")
print("Note: diagnostics are sample-based and do not modify the input file.")

Deduplicating edges_bidirectional.txt → edges_dedup.txt ...
Input : input/sustainability/version3/edges_bidirectional.txt  (49M)
Output: input/sustainability/version3/edges_dedup.txt
Running awk + sort deduplication ...
Done.  Unique edges written: 1677645

✓ Deduplicated file saved: input/sustainability/version3/edges_dedup.txt  (24.20 MB)
